# This one is gem
**To show every steps from the beginning as I need more practices**

## Run Ollama server 1st

In [1]:
import subprocess
subprocess.Popen(["ollama","serve"],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

<Popen: returncode: None args: ['ollama', 'serve']>

## Import Document --> Chunks/split 

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf = PyPDFLoader("paper.pdf").load()
doc = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(pdf)

print(len(doc))
print(f"Example of page_content: {doc[0].page_content}")
print(f"Example of metadata: {doc[0].metadata}")

55
Example of page_content: Home Surveillance System using Computer Vision 
and Convolutional Neural Network 
Xin Zhang, Won-Jae Yi and Jafar Saniie 
Embedded Computing and Signal Processing (ECASP) Research Laboratory ( http://ecasp.ece.iit.edu) 
Department of Electrical and Computer Engineering 
Illinois Institute of Technology, Chicago, Illinois, U.S.A.  
 
Abstract— In this paper, we introduce a home surveillance 
system that utilizes computer vision techniques to recognize
Example of metadata: {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText® 5.5.6 ©2000-2015 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 04/28/2019 12:45:25 AM'", 'creationdate': '2019-09-07T06:37:25+05:30', 'meeting starting date': '20 May 2019', 'moddate': '2020-10-02T11:22:28-04:00', 'ieee article id': '8833773', 'ieee issue id': '8833642', 'subject': '2019 IEEE International Conference on Electro Information Technology (EIT);2019; ; ;', 'application': "'Certif

## Import llm and embeddings

In [5]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="marin:latest")
embed = OllamaEmbeddings(model="nomic-embed-text:latest")


## Add data to Faiss vector store

In [9]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(doc,embed)

### Show datas of Faiss vector DB

In [13]:
result = vector_store.docstore._dict.values()

for a in result:
    print(a)

page_content='Home Surveillance System using Computer Vision 
and Convolutional Neural Network 
Xin Zhang, Won-Jae Yi and Jafar Saniie 
Embedded Computing and Signal Processing (ECASP) Research Laboratory ( http://ecasp.ece.iit.edu) 
Department of Electrical and Computer Engineering 
Illinois Institute of Technology, Chicago, Illinois, U.S.A.  
 
Abstract— In this paper, we introduce a home surveillance 
system that utilizes computer vision techniques to recognize' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText® 5.5.6 ©2000-2015 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 04/28/2019 12:45:25 AM'", 'creationdate': '2019-09-07T06:37:25+05:30', 'meeting starting date': '20 May 2019', 'moddate': '2020-10-02T11:22:28-04:00', 'ieee article id': '8833773', 'ieee issue id': '8833642', 'subject': '2019 IEEE International Conference on Electro Information Technology (EIT);2019; ; ;', 'application': "'Certified by IEEE PDFeXpress at

## Retriever 1 — Vector Retriever (Semantic Search)
What it does: Converts your **query into an embedding**, finds **chunks whose embeddings are closest to it in vector space.** *Finds meaning, not exact words.*
When to use: Works great when the user asks something conceptually related but uses different words. "How does the model focus?" would still find "attention mechanism" chunks.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k":4})
result_retrieve = retriever.invoke("What is vision in computer?")


print("Result Found:")
print(len(result_retrieve))

for i, doc in enumerate(result_retrieve):
    print(f"----chunk {i+1} (page {doc.metadata.get('page','?')})----")
    print(doc.page_content[:2000])
    print()

Result Found:
4
----chunk 1 (page 0)----
and principal component analysis (PCA) [4] can be used in 
many computer vision applications. Some common applications 
are facial recognition and scene recognition [4][5]. By 
distinguishing different objects using these techniques and 
applications, machines can interact with their environment to 
serve different needs. For instance, an embedded system with 
computer vision can detect intruders using recordings from 
surveillance cameras and warn users.

----chunk 2 (page 0)----
Home Surveillance System using Computer Vision 
and Convolutional Neural Network 
Xin Zhang, Won-Jae Yi and Jafar Saniie 
Embedded Computing and Signal Processing (ECASP) Research Laboratory ( http://ecasp.ece.iit.edu) 
Department of Electrical and Computer Engineering 
Illinois Institute of Technology, Chicago, Illinois, U.S.A.  
 
Abstract— In this paper, we introduce a home surveillance 
system that utilizes computer vision techniques to recognize

----chunk 3 (page

## Retriever 2 — Sparse Retriever (BM25 — Keyword Search)
What it does: Old-school keyword matching. Counts word frequencies (TF-IDF style). Finds chunks containing the exact words you typed. Fast, no embeddings needed. When to use: When you know the exact technical term. "BLEU score", "ATmega8", "DDRB register" — BM25 finds exact matches that semantic search sometimes misses.

In [ ]:
from langchain_community.retrievers import BM25Retriever

# It doesn't require embeddings — works directly
bm25 = BM25Retriever.from_documents(pdf)
bm25.k = 4

result = bm25.invoke("computer vision")

print("BM25 Result Found:")
print(len(result))
for i, doc in enumerate(result):
    print(f"----chunk {i+1} (page {doc.metadata.get('page','?')})----")
    print(doc.page_content[:500])
    print()

## Ensemble Retriever (Manual Implementation)

> ⚠️ `EnsembleRetriever` was removed in langchain-core >= 1.x.  
> We implement a simple weighted ensemble manually instead.

In [ ]:
from langchain_core.documents import Document
from typing import List

def ensemble_retrieve(
    query: str,
    retrievers: list,
    weights: list[float] | None = None,
    top_k: int = 4,
) -> List[Document]:
    """
    Simple weighted ensemble retriever.
    Combines results from multiple retrievers using weighted scores.

    Args:
        query: The search query.
        retrievers: List of retriever objects (must have .invoke()).
        weights: Weight for each retriever (must sum to 1.0).
                 Defaults to equal weights.
        top_k: Number of results to return.
    """
    if weights is None:
        weights = [1.0 / len(retrievers)] * len(retrievers)

    assert len(retrievers) == len(weights), "One weight per retriever required"
    assert abs(sum(weights) - 1.0) < 1e-6, "Weights must sum to 1.0"

    # Collect (doc, combined_score) pairs
    doc_scores: dict[tuple, tuple[Document, float]] = {}

    for retriever, weight in zip(retrievers, weights):
        docs = retriever.invoke(query)
        for doc in docs:
            # Use content + source + page as unique key
            key = (
                doc.page_content,
                doc.metadata.get("source", ""),
                doc.metadata.get("page", -1),
            )
            if key not in doc_scores:
                doc_scores[key] = (doc, 0.0)
            doc_scores[key] = (
                doc_scores[key][0],
                doc_scores[key][1] + weight,
            )

    # Sort by combined score (descending) and return top_k
    ranked = sorted(doc_scores.values(), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]


# --- Run Ensemble ---
ensemble_results = ensemble_retrieve(
    "What is vision in computer?",
    retrievers=[bm25, retriever],   # BM25 keyword + Vector semantic
    weights=[0.5, 0.5],
    top_k=4,
)

print(f"Ensemble returned {len(ensemble_results)} results:\n")
for i, doc in enumerate(ensemble_results):
    print(f"---- Result {i+1} (page {doc.metadata.get('page','?')}) ----")
    print(doc.page_content[:500])
    print()